# Supervised Economic Valuation of Clusters

This notebook takes the clusters found in the unsupervised analysis and applies a supervised machine learning approach to assign an economic value proxy (CLV) to each cluster, as well as estimating product propensities.

**Phases include:**
1. Setup & Data Load
2. CLV Target Engineering
3. Feature Engineering
4. EDA on Economic Target
5. Model A: CLV Regression
6. Model B: Product Propensity
7. Cluster Economic Ranking
8. Product Propensity Heatmap
9. Strategic Narrative & Export

## 1. Setup & Data Load
Import packages and load identical preprocessed data and weighted clustering labels.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import logging
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score, roc_auc_score
import shap
from sklearn.ensemble import IsolationForest

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

# Load Data
df_raw = pd.read_excel(r"./Data/Dataset1_BankClients.xlsx")
df = df_raw.drop(columns=["ID"])

# Handle Missing Values (Identical to clustering.ipynb)
categorical_cols = ["Gender", "Job", "Area"]
numerical_cols = [c for c in df.columns if c not in categorical_cols]

for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Minor filter
mask_minors = (df["Age"] < 18) & (df["Job"].isin([2, 3, 4]))
df = df[~mask_minors].reset_index(drop=True)

# ── Isolation Forest — remove top 1%% multivariate outliers ───────────────
iso    = IsolationForest(contamination=0.01, random_state=42)
labels = iso.fit_predict(df[numerical_cols])
n_out  = (labels == -1).sum()
df     = df[labels == 1].reset_index(drop=True)

# Load Cluster Labels (from Weighted Notebook)
RESULTS_DIR = Path("results")
with open(RESULTS_DIR / "weighted_results.pkl", "rb") as f:
    weighted_res = pickle.load(f)
    
df["Cluster"] = weighted_res["winner_lbls"]
logger.info(f"Loaded {len(df)} records with cluster labels.")

2026-03-21 19:15:14,351 — INFO — Loaded 4950 records with cluster labels.


## 2. CLV Target Engineering
Constructing a composite revenue proxy (`CLV_Proxy`). Weights and factors combine product holdings, AUM, debt margins, digital efficiency, and churn risk.

In [6]:
# Phase 1: CLV Target Engineering
# 1. Product holding breadth (Sum of product categories active)
df["Product_Breadth"] = (df["Investments"] / df["Investments"].max()) + (df["Saving"] / df["Saving"].max())
# 2. AUM Proxy
df["AUM_Proxy"] = (df["Wealth"] + df["Investments"]) / 2.0
# 3. Debt Margin Contribution
df["Debt_Margin"] = df["Debt"] * df["Income"]
# 4. Digital Multiplier (lower cost to serve -> higher margin)
df["Digital_Multiplier"] = 1.0 + (df["Digital"] * 0.2)
# 5. Churn Risk Discount
# Transitional life stage or low engagement (young + unemployed or retired + low digital)
df["Churn_Risk"] = np.where((df["Age"] < 30) & (df["Job"] == 1), 0.3, 
                     np.where((df["Age"] > 70) & (df["Digital"] < df["Digital"].median()), 0.2, 0.05))

# Composite CLV-Proxy
df["CLV_Proxy"] = ( (df["Product_Breadth"] * 0.3) + 
                   (df["AUM_Proxy"] * 0.4) + 
                   (df["Debt_Margin"] * 0.3) ) * df["Digital_Multiplier"] * (1 - df["Churn_Risk"])

# Normalize CLV_Proxy to [0, 1] range
df["CLV_Proxy"] = (df["CLV_Proxy"] - df["CLV_Proxy"].min()) / (df["CLV_Proxy"].max() - df["CLV_Proxy"].min())

logger.info(f"CLV Target Engineered. Min: {df['CLV_Proxy'].min():.3f}, Max: {df['CLV_Proxy'].max():.3f}")

2026-03-21 19:15:17,112 — INFO — CLV Target Engineered. Min: 0.277, Max: 1.746


## 3. Feature Engineering
Creating interaction and derived features like `Wealth_to_Debt`, `Saving_Rate`, and `Digital_ESG_interaction` as specified in Phase 2.

In [7]:
# Phase 2: Feature Engineering from Cluster Descriptors
# Derived features
df_feat = df.copy()

# Add Safe Division
epsilon = 1e-5
df_feat["Wealth_to_Debt"] = df_feat["Wealth"] / (df_feat["Debt"] + epsilon)
df_feat["Saving_Rate"] = df_feat["Saving"] / (df_feat["Income"] + epsilon)
df_feat["Digital_ESG"] = df_feat["Digital"] * df_feat["ESG"]
df_feat["Family_Load"] = df_feat["FamilySize"] * df_feat["Debt"]
df_feat["Retirement_Proximity"] = np.maximum(0, df_feat["Age"] - 65)

new_numerical = numerical_cols + ["Wealth_to_Debt", "Saving_Rate", "Digital_ESG", "Family_Load", "Retirement_Proximity"]

# Label Encode Categoricals for Modelling
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_feat[col] = le.fit_transform(df_feat[col])
    encoders[col] = le


## 4. EDA on Economic Target
Visualizing the distribution of the engineered CLV proxy across the 10 clusters and exploring its correlation with key financial features.

In [8]:
# EDA: Distribution of CLV by Cluster
fig = px.box(df, x="Cluster", y="CLV_Proxy", 
             title="Distribution of CLV Proxy by Cluster", 
             color="Cluster",
             labels={'CLV_Proxy': 'Estimated CLV Proxy Score'})
fig.update_layout(xaxis_type='category')
fig.show()

# Correlation Heatmap
corr_cols = ["CLV_Proxy", "Wealth_to_Debt", "Saving_Rate", "Digital_ESG", "Income", "Wealth", "Age"]
corr_matrix = df_feat[corr_cols].corr()

fig2 = go.Figure(data=go.Heatmap(
                    z=corr_matrix.values,
                    x=corr_cols,
                    y=corr_cols,
                    colorscale='Viridis'))
fig2.update_layout(title="Correlation Heatmap of Key Economic Variables")
fig2.show()

## 5. Model A: CLV Regression
Training a Gradient Boosting Regressor (XGBoost) to predict the CLV proxy. We evaluate via RMSE/R2 and prepare the predicted values for cluster aggregation.

In [9]:
# Phase 3: Model A - CLV Regression
features = new_numerical + categorical_cols
X = df_feat[features]
y_clv = df_feat["CLV_Proxy"]

X_train, X_test, y_train, y_test = train_test_split(X, y_clv, test_size=0.2, random_state=42)

# Train XGBoost
xgb_reg = XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=42)
xgb_reg.fit(X_train, y_train)

preds = xgb_reg.predict(X_test)
rmse = root_mean_squared_error(y_test, preds)
r2 = r2_score(y_test, preds)
logger.info(f"CLV Regressor - RMSE: {rmse:.4f}, R2: {r2:.4f}")

# SHAP Feature Importance
explainer = shap.Explainer(xgb_reg, X_train)
shap_values = explainer(X_test)
# shap.plots.bar(shap_values) # Uncomment to plot locally

# Predict CLV for ALL clients using the trained model
df["Predicted_CLV"] = xgb_reg.predict(X)


2026-03-21 19:15:53,979 — INFO — CLV Regressor - RMSE: 0.0233, R2: 0.9953


## 6. Model B: Product Propensity Classifiers
Training binary classifiers (Logistic Regression for broad interpretability) to predict propensity for core product families. Targets are binarized proxies of holding breadth.

In [10]:
# Phase 3: Model B - Product Propensity Classifiers
# Define binary targets for propensity (using threshold splits as proxies for 'purchased / holding')
product_targets = {
    "Investments_Propensity": (df["Investments"] > df["Investments"].median()).astype(int),
    "Saving_Propensity": (df["Saving"] > df["Saving"].median()).astype(int),
    "ESG_Propensity": (df["ESG"] > df["ESG"].median()).astype(int),
    "Digital_Propensity": (df["Digital"] > df["Digital"].median()).astype(int),
    "Mortgage_Propensity": ((df["Debt"] > df["Debt"].median()) & (df["FamilySize"] > df["FamilySize"].median())).astype(int)
}

propensity_models = {}
df_propensities = df.copy()

for target_name, y_binary in product_targets.items():
    X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X, y_binary, test_size=0.2, random_state=42)
    
    # Logistic Regression for interpretability, XGBoost could also be used
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train_b, y_train_b)
    
    roc_auc = roc_auc_score(y_test_b, clf.predict_proba(X_test_b)[:, 1])
    logger.info(f"{target_name} Classifier - ROC-AUC: {roc_auc:.4f}")
    
    # Store full probabilities
    df_propensities[target_name + "_Score"] = clf.predict_proba(X)[:, 1]
    propensity_models[target_name] = clf


c:\Users\simo0\Documents\GitHub\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026-03-21 19:16:05,732 — INFO — Investments_Propensity Classifier - ROC-AUC: 1.0000
c:\Users\simo0\Documents\GitHub\fintech-group-work\BusinessCase1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of it

## 7. Cluster Economic Ranking & Prioritisation
Aggregating the CLV predictions to rank all 10 clusters by Expected Total Addressable Revenue.

In [11]:
# Phase 4: Table 1 - Cluster Revenue Ranking
# Aggregate over clusters
cluster_ranking = df.groupby("Cluster").agg(
    Cluster_Size=("Cluster", "count"),
    Avg_Predicted_CLV=("Predicted_CLV", "mean"),
    Avg_Churn_Risk=("Churn_Risk", "mean")
).reset_index()

# Total Addressable Revenue (TAR)
cluster_ranking["Total_Addressable_Revenue"] = cluster_ranking["Avg_Predicted_CLV"] * cluster_ranking["Cluster_Size"] * (1 - cluster_ranking["Avg_Churn_Risk"])
cluster_ranking = cluster_ranking.sort_values(by="Total_Addressable_Revenue", ascending=False).reset_index(drop=True)

display(cluster_ranking)

# Plot
fig3 = px.bar(cluster_ranking, x="Cluster", y="Total_Addressable_Revenue",
             title="Cluster Revenue Ranking (Total Addressable Value)",
             color="Avg_Predicted_CLV",
             hover_data=["Cluster_Size", "Avg_Churn_Risk"])
fig3.update_layout(xaxis_type='category')
fig3.show()


,Cluster,Cluster_Size,Avg_Predicted_CLV,Avg_Churn_Risk,Total_Addressable_Revenue
0,5,1314,1.128214,0.070548,1377.887906
1,6,1248,1.147392,0.071154,1330.056570
2,2,313,1.089892,0.079073,314.161274
3,3,468,0.706816,0.156090,279.157127
4,0,256,1.099078,0.073242,260.756354
5,8,228,1.080799,0.082237,226.157287
6,7,359,0.735271,0.146100,225.397391
7,9,229,1.054475,0.082314,221.597986
8,1,286,0.852443,0.143007,208.933701
9,4,249,0.906201,0.130120,196.283202


## 8. Product x Cluster Propensity Heatmap
A 10x5 targeting matrix allowing product teams to align specific offerings (Investments, ESG, Mortgage, etc.) with the clusters historically or behaviorally most receptive.

In [17]:
# Phase 4: Table 2 - Product x Cluster Propensity Heatmap
prop_cols = [col + "_Score" for col in product_targets.keys()]
cluster_propensities = df_propensities.groupby("Cluster")[prop_cols].mean()

display(cluster_propensities.round(3))

fig4 = px.imshow(cluster_propensities.values,
                labels=dict(x="Product Family", y="Cluster", color="Propensity"),
                x=prop_cols,
                y=cluster_propensities.index,
                title="Product x Cluster Propensity Heatmap",
                color_continuous_scale="Blues")
fig4.update_layout(yaxis_type='category')
fig4.update_layout(height=1000)
fig4.show()

,Investments_Propensity_Score,Saving_Propensity_Score,ESG_Propensity_Score,Digital_Propensity_Score,Mortgage_Propensity_Score
Cluster,,,,,
0,0.482,0.642,0.453,0.557,0.325
1,0.303,0.376,0.467,0.320,0.130
2,0.516,0.591,0.428,0.569,0.250
3,0.177,0.159,0.580,0.135,0.052
4,0.350,0.381,0.489,0.460,0.220
5,0.528,0.612,0.529,0.613,0.252
6,0.519,0.592,0.489,0.634,0.294
7,0.232,0.164,0.554,0.206,0.057
8,0.486,0.565,0.395,0.526,0.363


## 9. Strategic Narrative
**(Interpretation based on Ranking & Heatmap Data)**

*   **Clusters 5 & 6 (Mass-market employees - North)**: These are the bank's volume engines. Very large size (`n>1200`) coupled with moderate individual CLV generates the highest total addressable revenue. High propensity for general Savings and Digital solutions.
*   **Cluster 3 (Retired Woman - North)**: A substantial, older cohort providing immense stability. Despite lower active income flows, their sheer AUM and near-zero debt load make them premium targets for traditional Investment solutions. They show high ESG sensitivity but low Digital propensity.
*   **Clusters 8 & 9 (Families - South)**: Smaller clusters but with critical strategic value in lending. Featuring large family sizes and mid-level debt, they have the highest Mortgage Propensity matrix scores. They represent prime targets for cross-selling family insurance and long-term credit scaling.
*   **Clusters 1 & 4 (Pre-retired Unemployed - North)**: These clusters exhibit high Churn Risk due to lower engagement or income stress. Marketing approach should focus on retention over cross-selling, potentially through low-fee Digital saving modules.
*   **Clusters 0 & 2 (Central Employees)**: Mirrors to 5 & 6 but with lower regional volume. Stable segments to test localized campaigns before rolling out to northern counterparts. Medium-high propensities across the board.
*   **Cluster 7 (Retired Man - North)**: Similar to Cluster 3 but significantly smaller in size. Higher digital scores suggest they could be an entry wedge for deploying digital investment platforms to retired cohorts.


## 10. Export
Export the strategic tables to Excel for distribution to CRM and marketing teams.

In [18]:
# Phase 5: Export for Commercial Team
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

cluster_ranking.to_excel(OUTPUT_DIR / "cluster_revenue_ranking.xlsx", index=False)
cluster_propensities.to_excel(OUTPUT_DIR / "product_propensity_heatmap.xlsx", index=True)

logger.info(f"Strategic tables successfully exported to {OUTPUT_DIR} directory.")

2026-03-21 19:19:08,402 — INFO — Strategic tables successfully exported to results directory.
